# DALL·E Image Generation

### Setup & imports

In [ ]:
# Setup & imports
from pathlib import Path
import ntpath
import pandas as pd
import re
import json
import os

In [ ]:
# DALL-E 3 requires version 1.0.0 or later of the openai-python library.

def dalle_api(desc):
    # You will need to set these environment variables or edit the following values.
    endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "")
    api_version = os.getenv("OPENAI_API_VERSION", "2024-04-01-preview")
    deployment = os.getenv("DEPLOYMENT_NAME", "Dalle3")
    api_key = os.getenv("AZURE_OPENAI_API_KEY", "")

    client = AzureOpenAI(
        api_version=api_version,
        azure_endpoint=endpoint,
        api_key=api_key,
    )

    result = client.images.generate(
        model=deployment,
        prompt="A medium-shot, cameraphone (2021) " + desc + "  Photo is Soft focus, shot on iPhone 12, for instant messaging in 2021.",
        n=1,
        style="vivid",
        quality="standard",
    )

    image_url = json.loads(result.model_dump_json())['data'][0]['url']
    return image_url

In [ ]:
# Data loading
QAs = []
gen = os.walk("./augmented/conversations")
next(gen)
for x in gen:
    directory = Path(x[0])
    text_files = list(directory.rglob('*.txt'))
    for file in text_files:
        dir = os.path.join("./augmented/images", file.parent.absolute().name)
        Path(dir).mkdir(parents=True, exist_ok=True)
        with open(file) as file:
            lines = [line.rstrip() for line in file]
            target_line = [s for s in lines if ": Image: [" in s]
            if len(target_line) > 0:
                match = re.search(r"\[[A-Za-z\s,-—.:\"\'\’!?\(\)&]+\]", target_line[0])
                try:
                    image_url = dalle_api(match.group(0)[1:-1])
                    img_data = requests.get(image_url).content
                    with open(os.path.join(dir, ntpath.split(file.name)[1])[:-4]+".jpg", "wb") as handler:
                        handler.write(img_data)
                except Exception as e:
                    with open(os.path.join(dir, ntpath.split(file.name)[1]), "w") as handler:
                        handler.write(e.message)
                

In [ ]:
# Data loading
answers = pd.read_json('./validate_answers.jsonl')
output = []

for index, answer in answers.iterrows():
    output.append(openai_api(answer['input'], answer['system']))